In [2]:
import os
import json
from datasets import load_dataset
from itertools import chain

In [84]:
taco_dataset = load_dataset('/home/kaixin/Desktop/mmcode/TACO')
print(taco_dataset["train"][0].keys())

dict_keys(['question', 'solutions', 'starter_code', 'input_output', 'difficulty', 'raw_tags', 'name', 'source', 'tags', 'skill_types', 'url', 'Expected Auxiliary Space', 'time_limit', 'date', 'picture_num', 'memory_limit', 'Expected Time Complexity'])


/home/kaixin/anaconda3/envs/mmcode/lib/python3.11/site-packages/datasets/load.py:922: FutureWarning: The repository for TACO contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at /home/kaixin/Desktop/mmcode/TACO/TACO.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


In [85]:
json.loads(taco_dataset["train"][1]["solutions"])

["INF = 10000000000.0\nmax_n = 50\nmax_k = 2000\n\ndef main():\n\t(n, s, k) = map(int, input().split())\n\ts -= 1\n\tbuf = [''] * (max_n + 1)\n\tdp = [[0 for i in range(max_n + 1)] for j in range(max_k + 1)]\n\tr = list(map(int, input().split()))\n\tc = input()\n\tanswer = INF\n\tfor i in range(len(c)):\n\t\tbuf[i] = c[i]\n\tfor i in range(k, -1, -1):\n\t\tfor j in range(n):\n\t\t\tdp[i][j] = INF\n\tfor j in range(n):\n\t\tvalue = abs(j - s)\n\t\tif k - r[j] <= 0:\n\t\t\tanswer = min(answer, value)\n\t\telse:\n\t\t\tdp[k - r[j]][j] = value\n\tfor i in range(k, 0, -1):\n\t\tfor j in range(n):\n\t\t\tif dp[i][j] < INF:\n\t\t\t\tfor l in range(n):\n\t\t\t\t\tif buf[j] != buf[l] and r[j] < r[l]:\n\t\t\t\t\t\tvalue = dp[i][j] + abs(j - l)\n\t\t\t\t\t\tif i - r[l] <= 0:\n\t\t\t\t\t\t\tanswer = min(answer, value)\n\t\t\t\t\t\telse:\n\t\t\t\t\t\t\tdp[i - r[l]][l] = min(dp[i - r[l]][l], value)\n\tif answer == INF:\n\t\tprint(-1)\n\t\treturn\n\tprint(answer)\n\ndef __starting_point():\n\tmain()\

In [89]:
taco_dataset_dict = {}

for index, item in enumerate(taco_dataset["train"]):
    identifier = f"train_{index}"
    spec = item["question"]
    taco_dataset_dict[identifier] = item

for index, item in enumerate(taco_dataset["test"]):
    identifier = f"test_{index}"
    spec = item["question"]
    taco_dataset_dict[identifier] = item
    
len(taco_dataset_dict)

26443

In [90]:
def get_taco_data(taco_id):
    if taco_id.startswith("train_"):
        real_id = int(taco_id[6:])
        return taco_dataset["train"][real_id]
    elif taco_id.startswith("test_"):
        real_id = int(taco_id[5:])
        return taco_dataset["test"][real_id]
    else:
        raise ValueError()

In [99]:
def get_problems_with_images(problems_folder):
    """
    Scan the image folder to retrieve problems with images.
    """
    # Create an empty list to store the tuples
    problems = []

    # Get all subdirectories with image folder
    subdirs = [os.path.join(problems_folder, d) for d in os.listdir(problems_folder) 
               if os.path.isdir(os.path.join(problems_folder, d)) and os.path.exists(os.path.join(problems_folder, d, "images"))]

    # Return the list of strs
    return subdirs

In [100]:
img_prob_codechef = get_problems_with_images("/home/kaixin/Desktop/mmcode/crawl/crawled/codechef/problems")
img_prob_g4g = get_problems_with_images("/home/kaixin/Desktop/mmcode/crawl/crawled/geeksforgeeks/problems")
img_prob_hackerrank = get_problems_with_images("/home/kaixin/Desktop/mmcode/crawl/crawled/hackerrank/problems")


In [105]:
# Copy files to new dataset
import shutil
import os
import json

destination_folder="/home/kaixin/Desktop/mmcode/mmcode_dataset"
for crawled_path in img_prob_hackerrank:
    with open(os.path.join(crawled_path, 'data.json'), 'r') as f:
        problem_data = json.load(f)
    taco_id = problem_data["taco_id"]

    # Use the last folder name of the crawled_path as the new folder name
    new_folder_name = os.path.basename(os.path.normpath(crawled_path))
    new_folder_path = os.path.join(destination_folder, new_folder_name)

    # Create the new folder
    os.makedirs(new_folder_path, exist_ok=True)

    # Copy files from crawled_path to the new folder
    shutil.copytree(crawled_path, new_folder_path, dirs_exist_ok=True)

    # write taco data to a file
    taco_data = get_taco_data(taco_id)
    taco_json_path = os.path.join(new_folder_path, "taco.json")
    with open(taco_json_path, 'w') as f:
        json.dump(taco_data, f, indent=4)

    

In [3]:
# Merge crawled data and taco data
destination_folder="/home/kaixin/Desktop/mmcode/mmcode_dataset_taco"
subdirs = [os.path.join(destination_folder, d) for d in os.listdir(destination_folder) 
               if os.path.isdir(os.path.join(destination_folder, d))]
for subdir in subdirs:
    # Paths for taco.json and data.json in the current subdir
    taco_json_path = os.path.join(subdir, 'taco.json')
    data_json_path = os.path.join(subdir, 'data.json')
    new_data_json_path = os.path.join(subdir, 'new_data.json')

    # Initialize a dictionary to hold the combined data
    combined_data = {}

    # Read and add the content of taco.json to combined_data
    if os.path.exists(taco_json_path):
        with open(taco_json_path, 'r') as file:
            taco_data = json.load(file)
            combined_data.update(taco_data)

    # Read and add the content of data.json under "crawled_meta"
    if os.path.exists(data_json_path):
        with open(data_json_path, 'r') as file:
            data = json.load(file)
            combined_data['crawled_meta'] = data

    # Write the combined data to the new data.json file
    with open(new_data_json_path, 'w') as file:
        json.dump(combined_data, file, indent=4)